In [ ]:
import pandas as pd
import os
from datasets.loader import save_splits
from datasets.paths import ProjectPaths


In [ ]:
paths = ProjectPaths()


In [ ]:
splits = ["train", "dev", "test"]

dfs = []
for split in splits:
	path = os.path.join(paths.raw_dir, f"stsb-es-{split}.csv",)
	df = pd.read_csv(
		path,
		names=["sentence1", "sentence2", "score"]
	)
	df["split"] = split
	dfs.append(df)

df = pd.concat(dfs, ignore_index=True)


In [4]:
df["score_norm"] = df["score"] / 5


In [5]:
display(df)


,sentence1,sentence2,score,split,score_norm
0,Un avión está despegando.,Un avión está despegando.,5.00,train,1.00
1,Un hombre está tocando una gran flauta.,Un hombre está tocando una flauta.,3.80,train,0.76
2,Un hombre está untando queso rallado en una pi...,Un hombre está untando queso rallado en una pi...,3.80,train,0.76
3,Tres hombres están jugando al ajedrez.,Dos hombres están jugando al ajedrez.,2.60,train,0.52
4,Un hombre está tocando el violonchelo.,Un hombre sentado está tocando el violonchelo.,4.25,train,0.85
...,...,...,...,...,...
8623,Filipinas y el Canadá se comprometen a seguir ...,Filipinas salva 100 tras el hundimiento de un ...,0.00,test,0.00
8624,Israel impide a los palestinos entrar en la Ci...,La solución de dos estados entre los palestino...,1.00,test,0.20
8625,¿Cuánto sabes del Servicio Secreto?,Los legisladores de ambos lados expresan su in...,1.00,test,0.20
8626,Obama lucha por calmar los temores de los saud...,Myanmar lucha por finalizar las listas de vota...,0.00,test,0.00


In [6]:
df.groupby("split")["score_norm"].describe()


,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
dev,1500.0,0.472782,0.300097,0.0,0.200,0.51,0.72,1.0
test,1379.0,0.521583,0.305103,0.0,0.263,0.56,0.76,1.0
train,5749.0,0.540200,0.292880,0.0,0.300,0.60,0.76,1.0


In [7]:
def is_valid_text(sentence: str):
    if not isinstance(sentence, str):
        return False
    
    sentence = sentence.strip()
    if len(sentence) <= 0:
        return False

    try:
        sentence.encode("cp1252")
    except UnicodeEncodeError:
        return False

    return True


def clean_dataframe(df: pd.DataFrame):
    valid_rows = []

    for idx, row in df.iterrows():
        sentence1 = row["sentence1"]
        sentence2 = row["sentence2"]
        split = row.get("split", "unknown")

        if is_valid_text(sentence1) and is_valid_text(sentence2):
            valid_rows.append(row)
        else:
            print(f"Removing row {idx} (split={split}):")
            print("  sentence1:", sentence1)
            print("  sentence2:", sentence2)

    return pd.DataFrame(valid_rows).reset_index(drop=True)

df = clean_dataframe(df)


Removing row 5006 (split=train):
  sentence1: Sienna Miller testifica en el juicio por piratería telefónica en el Reino Unido
  sentence2: Sienna Miller ataca a la prensa por 'Äėtitillating'Äô informes en el juicio de hacking
Removing row 5010 (split=train):
  sentence1: 'ÄėGlee'Äô estrella Cory Monteith fue encontrado muerto en una habitación de hotel
  sentence2: Cory Monteith fue encontrado muerto: La estrella canadiense 'ÄėGlee'Äô tenía 31 años
Removing row 5176 (split=train):
  sentence1: Noticias de Terrorismo y el Conflicto Israelí-Palestino (22 de enero 'Äď 28, 2014)
  sentence2: Noticias de Terrorismo y el Conflicto Israelí-Palestino (31 de julio 'Äď 6 de agosto de 2013)
Removing row 5239 (split=train):
  sentence1: Corea del Norte: los expertos piden un diálogo 'Äď y dicen que China debe desempeñar un papel
  sentence2: Corea del Norte corta los últimos lazos con el Sur al prohibir el parque industrial conjunto
Removing row 5243 (split=train):
  sentence1: Mandela volvió al h

In [ ]:
save_splits(df, paths.processed_dir)

	